# Create a forcing wave file for CICE6 standalone

This notebook produces a wave forcing dataset given CICE6-WIM and CAWCR outputs. CICE6-WIM data is taken within the sea ice cover and CAWCR is applied in the open ocean. A Bretschneider spectrum is assumed for all cells (including those within the ice).

$$S_f[i_{\text{freq}}, :, :, :] = \frac{5}{16} H_s^2 T_p^{-4} f_{\text{global}}[i_{\text{freq}}, :, :, :]^{-5} \exp\left( -\frac{5}{4} \left( f_{\text{global}}[i_{\text{freq}}, :, :, :] T_p \right)^{-4} \right)$$


Check the original notebook at `/Users/noahday/GitHub/cice-dev/regridding/wave_spectrum_forcing.ipynb`.

In [ ]:
# Check the variables names in the WW3 forcing given by CICE consortium
import xarray as xr
import numpy as np
import pandas as pd
import os
import subprocess
import matplotlib.pyplot as plt

# Import my functions
functions_path = os.path.abspath("/home/566/nd0349/access-om3-analysis/functions")
if functions_path not in sys.path:
    sys.path.append(functions_path)
from get_files import *
from plot_settings import *
from fstd import *
from attenuation_models import *
test()

url = "https://zenodo.org/records/3728360/files/CICE_data_gx3_forcing_WW3-20200320.tar.gz?download=1"
file_path = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/CICE_data_gx3_forcing_WW3-20200320.tar.gz"
target_dir = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3"

if not os.path.exists(file_path):
    print(f"Downloading {file_path}...")
    subprocess.run(["wget", url, "-O", file_path], check=True)
    subprocess.run(["tar", "-xzvf", os.path.basename(file_path)], cwd=target_dir, check=True)
    src_file = os.path.join(target_dir, "CICE_data", "forcing", "gx3", "WW3", "ww3.20100101_efreq_remapgx3.nc")
    dst_file = os.path.join(target_dir, "ww3.20100101_efreq_remapgx3.nc")
    subprocess.run(["cp", src_file, dst_file], check=True)
    print(f"Copied {src_file} to {dst_file}")
else:
    print(f"{file_path} already exists. Skipping download.")


# ds_ww3.encoding

In [ ]:
file_path = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_ww3 = xr.open_dataset(file_path, engine="netcdf4")
ds_ww3

CICE6 standalone forcing requires a NetCDF file with dimensions `(time, nj, ni, f)`. The spectrum data is named `efreq` and should have dimensions `(time, f, nj, ni)`.

In [ ]:
# Load the NetCDF file from CICE6-WIM
file_path = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/ice_output/iceh.2019-01-01.nc"
date_str = file_path.split("iceh.")[1].split(".nc")[0]
ds = xr.open_dataset(file_path)

# Extract variables
Hs = ds["wave_sig_ht"]  # Significant wave height (time, nj, ni)
Tp = ds["peak_period"]  # Peak wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs.time)
nj = len(Hs.nj)
ni = len(Hs.ni)

fmin = 0.042  # Minimum frequency (Hz)
fmax = 0.4          # Maximum frequency (Hz)

# Angular frequencies (rad/s)
om1 = 2 * np.pi * fmin
om2 = 2 * np.pi * fmax

# Frequency step size in rad/s
om_0 = (om2 - om1) / (nw - 1)

# Define frequency range (in Hz) based on omega values
omega = np.linspace(om1, om2, nw)  # Angular frequencies
f = np.array([[[omega / (2 * np.pi)]]])

# df = f - np.concatenate(([0], f[0:-1]), axis=0)
df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)

f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)

# Now we need to ensure that the `f` coordinate is broadcast to match the shape of the data
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)

# Compute the Bretschneider spectrum
S_f = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f[i_freq, :, :, :] = (5 / 16) * (Hs ** 2) * Tp**(-4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] * Tp) ** (-4))

# Convert to xarray DataArray
S_f_xr = xr.DataArray(
    S_f,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="efreq"
)


df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f
# Sum over the first dimension (axis 0) to aggregate
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="Hs_test"
)


In [ ]:
# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs.where(ds['TLAT'] < -30, drop=True).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_xr - Hs
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()
plt.show()


## MOM6-CICE6-WW3 outputs

In [ ]:
# ds_om3["HS"]

In [ ]:
file_path_om3 = '/scratch/tm70/ek4684/access-om3/archive/MCW_100km_jra_iaf_IC4_KPP/output000/access-om3.ww3.hi.1958-01-02-00000.nc'
# '/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/wave_output/access-om3.ww3.hi.1958-01-02-00000.nc'

ds_om3 = xr.open_dataset(file_path_om3)
# ds_om3 = ds_om3.isel(time=slice(-2,-1)) # select middle day (closest to average?)
ds_om3 = ds_om3.rename({"ny": "nj", "nx": "ni"})

Hs_om3 = ds_om3["HS"]  # Significant wave height (time, nj, ni)
Tp_om3 = ds_om3["T01"]  # Mean wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = max(len(Hs_om3.time), 1)
nj = len(Hs_om3.nj)
ni = len(Hs_om3.ni)

df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)

# Compute the Bretschneider spectrum
S_f_om3 = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f[i_freq, :, :, :] = (5 / 16) * (Hs_om3 ** 2) * Tp_om3**(-4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] * Tp_om3) ** (-4))
    print(np.nanmax(S_f_om3[i_freq, :, :, :]))

# Convert to xarray DataArray
S_f_cawcr_xr = xr.DataArray(
    S_f_om3,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds_om3["time"], "nj": ds_om3["nj"], "ni": ds_om3["ni"]},
    name="efreq"
)


df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_om3
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_om3_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_om3["time"], "nj": ds_om3["nj"], "ni": ds_om3["ni"]},
    name="Hs_test"
)



# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_om3_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_om3.isel(nj=slice(0,100)).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_om3_xr - Hs_om3
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()

plt.show()

S_f_om3_xr = S_f_om3_xr.assign_coords(time=S_f_om3_xr.time + pd.Timedelta(hours=2))

S_f_combined_xr = xr.where(S_f_xr == 0, S_f_om3_xr, S_f_xr)
S_f_combined_xr

In [ ]:
# ((5 / 16) * (Hs_om3 ** 2) * Tp_om3**(-4) * f_global[i_freq, :, :, :]**(-5)).shape

In [ ]:
# ds_om3["EF"]

# Regrid the spectrum from WW3 to CICE6

In [ ]:
example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_example = xr.open_dataset(example_file)
ds_example

In [ ]:
# ds_example.time

In [ ]:
# ds_om3.f

In [ ]:
# 

In [ ]:
file_path_om3 = '/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_iaf_2010/output000/access-om3.ww3.hi.2010-01-02-00000.nc'
# file_path_om3 = '/scratch/tm70/ek4684/access-om3/archive/MCW_100km_jra_iaf_IC4_KPP/output000/access-om3.ww3.hi.1958-01-02-00000.nc'
# '/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/wave_output/access-om3.ww3.hi.1958-01-02-00000.nc'

ds_om3 = xr.open_dataset(file_path_om3)
ds_om3 = ds_om3.rename({"ny": "nj", 
                        "nx": "ni", 
                        "freq": "f",
                        "lon": "TLON",
                        "lat": "TLAT",
                        "EF": "efreq",
                        "HS": "wave_sig_ht",
                        "FP0": "peak_period"
                       })
ds_om3

In [ ]:
freqs, periods = get_ww3_freqs(nk=25)
xr_freqs = xr.DataArray(
    freqs,
    dims=("f",),
    # coords={"f": ds_om3.f},
    name='f',
    attrs={
        'units': 's-1',
        'description': 'Wave frequency',
        'long_name': 'wave_frequency',
    }
)
xr_freqs


keep_vars = ["TLON", "TLAT", "wave_sig_ht", "peak_period", "efreq", "f"]

ds_om3 = ds_om3.drop_vars(
    [v for v in ds_om3.data_vars if v not in keep_vars],
    errors="ignore",
)
ds_om3["f"] = xr_freqs
ds_om3

In [ ]:
ds_om3.time
year = ds_om3.time.dt.year.item()
month = ds_om3.time.dt.month.item()
day = ds_om3.time.dt.day.item()
print(year)

import numpy as np

def make_6hourly_times(year, month, day, endpoint=False):
    base = np.datetime64(f"{year}-{month:02d}-{day:02d}T00:00:00", "ns")
    # offsets = np.arange(0, 24, 6).astype("timedelta64[h]")
    offsets = np.linspace(0, 24, 4, endpoint=endpoint).astype("timedelta64[h]")
    times_ns = base + offsets
    return times_ns

times = make_6hourly_times(year, month, day, endpoint=True)
times

In [ ]:
da

In [ ]:
# np.broadcast_to(ds_om3, (nw, 4, nj, ni))
da = ds_om3

In [ ]:
# Create a dummy DataArray with the time dimension you want
dummy_time = xr.DataArray(
    np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),  # shape: ("time", "f", "nj", "ni")
    coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
    dims=("time", "f", "nj", "ni"),
)

# Broadcast efreq along the time dimension
da_broadcast = xr.broadcast(dummy_time, da.drop_vars(['TLON', 'TLAT']))[1]  # second element is original data broadcasted
da_broadcast['TLON'] = da['TLON']
da_broadcast['TLAT'] = da['TLAT']
da_broadcast

In [ ]:
da_broadcast['wave_sig_ht'].isel(time=0).plot()

In [ ]:
dummy_txy = xr.DataArray(
    np.empty((len(times), len(da.nj), len(da.ni))),
    coords={"time": times, "nj": da.nj, "ni": da.ni},
    dims=("time", "nj", "ni"),
)

vars_no_f = ["wave_sig_ht", "peak_period"]

da_tf = xr.broadcast(
    dummy_txy,
    da[vars_no_f]
)[1]

In [ ]:
dummy_tfxy = xr.DataArray(
    np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),
    coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
    dims=("time", "f", "nj", "ni"),
)

efreq_bt = xr.broadcast(dummy_tfxy, da["efreq"])[1]

In [ ]:
da_broadcast = xr.Dataset(
    {
        **da_tf.data_vars,
        "efreq": efreq_bt,
        "TLON": da["TLON"],
        "TLAT": da["TLAT"],
    },
    coords={
        "time": times,
        "f": da.f,
        "nj": da.nj,
        "ni": da.ni,
    },
)

In [ ]:
da_broadcast

In [ ]:
# da_broadcast['time'] = ds_example['time']

In [ ]:
# da_broadcast.time

In [ ]:
# ds_example.time

In [ ]:
# from netCDF4 import Dataset
# import netCDF4 as nc

# ds_example = Dataset(example_file, "r")

# # Create a new NetCDF file
# new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}-{day-1:02d}_test.nc"
# ds_new = Dataset(new_file, "w", format="NETCDF4")

# # Copy global attributes
# ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})

# # Define new dimensions based on S_f_combined_ds
# # for dim_name, size in da_broadcast.sizes.items():
# #     ds_new.createDimension(dim_name, size)

# ds_new.createDimension("time", None)  # UNLIMITED
# ds_new.createDimension("ni", da_broadcast.sizes["ni"])
# ds_new.createDimension("nj", da_broadcast.sizes["nj"])
# ds_new.createDimension("f",  da_broadcast.sizes["f"])

# time_var = ds_new.createVariable("time", "f8", ("time",), fill_value=False)
# time_var.standard_name = "time"
# time_var.long_name = "julian day (UT)"
# time_var.units = "days since 1990-01-01 00:00:00"
# time_var.calendar = "standard"
# time_var.axis = "T"


# time_vals = da_broadcast["time"].values

# # time_var[:] = nc.date2num(
# #     time_vals.astype("datetime64[ns]").astype(object),  # → python datetime
# #     units=time_var.units,
# #     calendar=time_var.calendar,
# # )

# f_var = ds_new.createVariable("f", "f4", ("f",), fill_value=False)
# f_var.long_name = "wave_frequency"
# f_var.units = "s-1"
# f_var.axis = "Hz"
# f_var.standard_name = "wave_frequency"

# f_var[:] = da_broadcast["f"].values.astype("float32")

# TLON = ds_new.createVariable("TLON", "f4", ("nj", "ni"), fill_value=False)
# TLON.standard_name = "longitude"
# TLON.long_name = "longitude"
# TLON.units = "degrees_east"
# TLON._CoordinateAxisType = "Lon"
# TLON[:] = da_broadcast["TLON"].values.astype("float32")

# TLAT = ds_new.createVariable("TLAT", "f4", ("nj", "ni"), fill_value=False)
# TLAT.standard_name = "latitude"
# TLAT.long_name = "latitude"
# TLAT.units = "degrees_north"
# TLAT._CoordinateAxisType = "Lat"
# TLAT[:] = da_broadcast["TLAT"].values.astype("float32")

# # for var_name, da_var in da_broadcast.data_vars.items():  # iterate over DataArray vars
# for var_name, var in da_broadcast.variables.items():
#     if var_name in da_broadcast:
#         if var_name in ("efreq"):
#         #     new_var = ds_new.createVariable(var_name, var.dtype, var.dims)
        
#         #     # Copy variable attributes
#         #     new_var.setncatts({attr: var.attrs.get(attr) for attr in var.attrs})
        
#         #     new_var[:] = da_broadcast.transpose("time", "nj", "ni")[var_name].values
            
#         # elif var_name in ("efreq"):
#             # dims = da_var.dims  # these are correct: ('time','f','nj','ni')
#             new_var = ds_new.createVariable(var_name, var.dtype, var.dims)
        
#             # Copy variable attributes
#             new_var.setncatts({attr: var.attrs.get(attr) for attr in var.attrs})
        
#             new_var[:] = da_broadcast.transpose("time", "f", "nj", "ni")[var_name].values
#         # else:
#         #     new_var = ds_new.createVariable(var_name, var.dtype, var.dims)
        
#         #     # Copy variable attributes
#         #     new_var.setncatts({attr: var.attrs.get(attr) for attr in var.attrs})
        
#         #     new_var[:] = da_broadcast[var_name].values
            
#             # copy attributes if you have them
#             # for attr in da_var.attrs:
#             #     new_var.setncattr(attr, da_var.attrs[attr])
#             # # assign values
#             # new_var[:] = da_var.values

# # Close the datasets
# ds_example.close()
# ds_new.close()

# print(f"New NetCDF file '{new_file}' created successfully with data from da_broadcast!")

In [ ]:
# da_broadcast['wave_sig_ht']#.isel(time=0)

In [ ]:
from netCDF4 import Dataset
import netCDF4 as nc
import numpy as np
import cftime

ds_example = Dataset(example_file, "r")

new_file = (
    f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/"
    f"cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}-{day-1:02d}_test.nc"
)
ds_new = Dataset(new_file, "w", format="NETCDF4")

# ------------------------------------------------------------------
# Global attributes
# ------------------------------------------------------------------
ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})

# ------------------------------------------------------------------
# Dimensions (time must be unlimited)
# ------------------------------------------------------------------
ds_new.createDimension("time", None)
ds_new.createDimension("f",  da_broadcast.sizes["f"])
ds_new.createDimension("nj", da_broadcast.sizes["nj"])
ds_new.createDimension("ni", da_broadcast.sizes["ni"])

# ------------------------------------------------------------------
# time (double, CF-compliant)
# ------------------------------------------------------------------
time_var = ds_new.createVariable("time", "f8", ("time",), fill_value=False)
time_var.standard_name = "time"
time_var.long_name = "julian day (UT)"
time_var.units = "days since 1990-01-01 00:00:00"
time_var.calendar = "standard"
time_var.axis = "T"

time_vals = da_broadcast["time"].values
times_py = pd.to_datetime(times).to_pydatetime()

# Now use date2num
time_nums = cftime.date2num(times_py, 
                            units='hours since 1900-01-01 00:00:00', 
                            calendar='gregorian')

time_var[:] = time_nums

# ------------------------------------------------------------------
# f (frequency axis)
# ------------------------------------------------------------------
f_var = ds_new.createVariable("f", "f4", ("f",), fill_value=False)
f_var.long_name = "wave_frequency"
f_var.units = "s-1"
f_var.axis = "Hz"
f_var.standard_name = "wave_frequency"
f_var[:] = da_broadcast["f"].values.astype("float32")

# ------------------------------------------------------------------
# Grid (2-D only — collapse any broadcast dims)
# ------------------------------------------------------------------
TLON = ds_new.createVariable("TLON", "f4", ("nj", "ni"), fill_value=False)
TLON.standard_name = "longitude"
TLON.long_name = "longitude"
TLON.units = "degrees_east"
TLON._CoordinateAxisType = "Lon"
TLON[:] = da_broadcast["TLON"].isel(
    {d: 0 for d in da_broadcast["TLON"].dims if d not in ("nj", "ni")}
).values.astype("float32")

TLAT = ds_new.createVariable("TLAT", "f4", ("nj", "ni"), fill_value=False)
TLAT.standard_name = "latitude"
TLAT.long_name = "latitude"
TLAT.units = "degrees_north"
TLAT._CoordinateAxisType = "Lat"
TLAT[:] = da_broadcast["TLAT"].isel(
    {d: 0 for d in da_broadcast["TLAT"].dims if d not in ("nj", "ni")}
).values.astype("float32")

# ------------------------------------------------------------------
# wave_sig_ht(time, nj, ni)
# ------------------------------------------------------------------
wave_sig_ht = da_broadcast["wave_sig_ht"]

wave_sig_ht_vals = da_broadcast["wave_sig_ht"].transpose("time","nj","ni").values.astype("float32")

# replace NaNs/Infs/extreme fill values
wave_sig_ht_vals[~np.isfinite(wave_sig_ht_vals)] = 0.0
wave_sig_ht_vals[wave_sig_ht_vals < 0] = 0.0  # optional, since spectra are non-negative

wave_sig_ht_var = ds_new.createVariable(
    "wave_sig_ht",
    "f4",
    ("time","nj","ni"),
    chunksizes=(1,300,360),
    fill_value=False,
)
wave_sig_ht_var[:] = wave_sig_ht_vals
wave_sig_ht_var.setncatts(da_broadcast["wave_sig_ht"].attrs)

# Peak period
peak_period = da_broadcast["peak_period"]

peak_period_vals = da_broadcast["peak_period"].transpose("time","nj","ni").values.astype("float32")

# replace NaNs/Infs/extreme fill values
peak_period_vals[~np.isfinite(peak_period_vals)] = 0.0
peak_period_vals[peak_period_vals < 0] = 0.0  # optional, since spectra are non-negative

peak_period_var = ds_new.createVariable(
    "peak_period",
    "f4",
    ("time","nj","ni"),
    chunksizes=(1,300,360),
    fill_value=False,
)
peak_period_var[:] = peak_period_vals
peak_period_var.setncatts(da_broadcast["peak_period"].attrs)

# ------------------------------------------------------------------
# efreq(time, f, nj, ni)
# ------------------------------------------------------------------
efreq = da_broadcast["efreq"]

efreq_vals = da_broadcast["efreq"].transpose("time","f","nj","ni").values.astype("float32")

# replace NaNs/Infs/extreme fill values
efreq_vals[~np.isfinite(efreq_vals)] = 0.0
efreq_vals[efreq_vals < 0] = 0.0  # optional, since spectra are non-negative

efreq_var = ds_new.createVariable(
    "efreq",
    "f4",
    ("time","f","nj","ni"),
    chunksizes=(1,1,300,360),
    fill_value=False,
)
efreq_var[:] = efreq_vals
efreq_var.setncatts(da_broadcast["efreq"].attrs)

# ------------------------------------------------------------------
# Close files
# ------------------------------------------------------------------
ds_example.close()
ds_new.close()

print(f"New NetCDF file '{new_file}' created successfully.")

In [ ]:
ds_tmp = xr.open_dataset(new_file)
ds_tmp

In [ ]:
ds_tmp['wave_sig_ht'].isel(time=1).plot()

In [ ]:
# Create a dummy DataArray with the time dimension you want
dummy_time = xr.DataArray(
    np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),  # shape: ("time", "f", "nj", "ni")
    coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
    dims=("time", "f", "nj", "ni"),
)

# Broadcast efreq along the time dimension
da_broadcast = xr.broadcast(dummy_time, da.drop_vars(['TLON', 'TLAT']))[1]  # second element is original data broadcasted
da_broadcast['TLON'] = da['TLON']
da_broadcast['TLAT'] = da['TLAT']
da_broadcast

dummy_txy = xr.DataArray(
    np.empty((len(times), len(da.nj), len(da.ni))),
    coords={"time": times, "nj": da.nj, "ni": da.ni},
    dims=("time", "nj", "ni"),
)

vars_no_f = ["wave_sig_ht", "peak_period"]

da_tf = xr.broadcast(
    dummy_txy,
    da[vars_no_f]
)[1]

dummy_tfxy = xr.DataArray(
    np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),
    coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
    dims=("time", "f", "nj", "ni"),
)

efreq_bt = xr.broadcast(dummy_tfxy, da["efreq"])[1]


da_broadcast = xr.Dataset(
    {
        **da_tf.data_vars,
        "efreq": efreq_bt,
        "TLON": da["TLON"],
        "TLAT": da["TLAT"],
    },
    coords={
        "time": times,
        "f": da.f,
        "nj": da.nj,
        "ni": da.ni,
    },
)

In [ ]:
da_broadcast

In [ ]:
da['wave_sig_ht']#.plot()

In [ ]:
da4 = xr.concat([da['wave_sig_ht']] * 4, dim='time')
da4

In [ ]:
da4 = xr.concat([da['efreq']] * 4, dim='time')
da4

In [ ]:
da4.isel(time=-1).plot()

In [ ]:
da4.time

In [ ]:
xr.concat([da['efreq']] * npoints, dim='time')

In [ ]:
def year_month_to_output(year, month, base_year=2010):
    """
    Map (year, month) -> output index.
    
    month: 1–12
    """
    if not 1 <= month <= 12:
        raise ValueError("month must be in 1..12")

    return 12 * (year - base_year) + (month - 1)

In [ ]:
output_dir = f"output{year_month_to_output(year, month, base_year=2010):03d}"
    

    # ---- file is offset by +1 day ----
file_date = date(year, month, day) + timedelta(days=1)

file_path_om3 = file_path_om3_template.format(
    output_dir=output_dir,
    year=file_date.year,
    month=file_date.month,
    day=file_date.day
)
file_path_om3

# output_dir

# Create a whole month of forcing

In [ ]:
import calendar
import pandas as pd
import numpy as np
import xarray as xr
import cftime
from netCDF4 import Dataset
from datetime import date, timedelta

def make_8hourly_times(year, month, day, endpoint=False):
    base = np.datetime64(f"{year}-{month:02d}-{day:02d}T00:00:00", "ns")
    # offsets = np.arange(0, 24, 6).astype("timedelta64[h]")
    offsets = np.linspace(0, 24, 4, endpoint=endpoint).astype("timedelta64[h]")
    times_ns = base + offsets
    return times_ns

# Example: month to process
year = 2010
month = 12

# Number of days in the month
n_days = calendar.monthrange(year, month)[1]

# Open your daily file template
# output_month = year_month_to_output(year, month, base_year=2010)
file_path_om3_template = "/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_iaf_2010/{output_dir}/access-om3.ww3.hi.{year:04d}-{month:02d}-{day:02d}-00000.nc"



# Prepare new NetCDF output
new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}.nc"
ds_new = Dataset(new_file, "w", format="NETCDF4")

ds_example_roach = Dataset(example_file, "r")
ds_new.setncatts({attr: ds_example_roach.getncattr(attr) for attr in ds_example_roach.ncattrs()})

# Create dimensions (time unlimited)
ds_new.createDimension("time", None)

# We'll use first day to get static dimensions
ds_example = Dataset('/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_2010-01-01_test.nc', "r")
ds_new.createDimension("f", ds_example.dimensions["f"].size)
ds_new.createDimension("nj", ds_example.dimensions["nj"].size)
ds_new.createDimension("ni", ds_example.dimensions["ni"].size)

# Create variables (simplified; you'll fill attributes as before)
time_var = ds_new.createVariable("time", "f8", ("time",), fill_value=False)
time_var.units = "hours since 1900-01-01 00:00:00"
time_var.calendar = "gregorian"

# Fill static variables from first day
# ------------------------------------------------------------------
# f (frequency axis)
# ------------------------------------------------------------------
# Create variable once
f_var = ds_new.createVariable("f", "f4", ("f",))
freqs, periods = get_ww3_freqs(nk=25)
f_var[:] = freqs.astype("float32")
#da_broadcast["f"].values.astype("float32")  # assign values

# Set attributes
f_var.long_name = "wave_frequency"
f_var.units = "s-1"
f_var.axis = "Hz"
f_var.standard_name = "wave_frequency"

# ------------------------------------------------------------------
# Grid (2-D only — collapse any broadcast dims)
# ------------------------------------------------------------------
TLON = ds_new.createVariable("TLON", "f4", ("nj", "ni"), fill_value=False)
TLON.standard_name = "longitude"
TLON.long_name = "longitude"
TLON.units = "degrees_east"
TLON._CoordinateAxisType = "Lon"
TLON[:] = ds_example.variables["TLON"][:]
# da_broadcast["TLON"].isel(
#     {d: 0 for d in da_broadcast["TLON"].dims if d not in ("nj", "ni")}
# ).values.astype("float32")

TLAT = ds_new.createVariable("TLAT", "f4", ("nj", "ni"), fill_value=False)
TLAT.standard_name = "latitude"
TLAT.long_name = "latitude"
TLAT.units = "degrees_north"
TLAT._CoordinateAxisType = "Lat"
TLAT[:] = ds_example.variables["TLAT"][:]


wave_sig_ht_var = ds_new.createVariable(
    "wave_sig_ht",
    "f4",
    ("time", "nj", "ni"),
    chunksizes=(1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)
peak_period_var = ds_new.createVariable(
    "peak_period",
    "f4",
    ("time", "nj", "ni"),
    chunksizes=(1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)
efreq_var = ds_new.createVariable(
    "efreq",
    "f4",
    ("time", "f", "nj", "ni"),
    chunksizes=(1, 1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)



# Prepare time accumulator
all_times = []

# Track current time index in unlimited time dimension
time_index = 0

for day in tqdm(range(1, n_days + 1)):
    # file_path_om3 = file_path_om3_template.format(year=year, month=month, day=day)
    output_dir = f"output{year_month_to_output(year, month, base_year=2010):03d}"
    

        # ---- file is offset by +1 day ----
    file_date = date(year, month, day) + timedelta(days=1)
    
    file_path_om3 = file_path_om3_template.format(
        output_dir=output_dir,
        year=file_date.year,
        month=file_date.month,
        day=file_date.day
    )
    ds_om3 = xr.open_dataset(file_path_om3)
    
    # Rename and keep only variables you need (same as your single-day workflow)
    ds_om3 = ds_om3.rename({
        "ny": "nj", "nx": "ni", "freq": "f",
        "lon": "TLON", "lat": "TLAT",
        "EF": "efreq", "HS": "wave_sig_ht", "FP0": "peak_period"
    })
    keep_vars = ["TLON","TLAT","wave_sig_ht","peak_period","efreq","f"]
    ds_om3 = ds_om3[keep_vars]


    da = ds_om3

        # Make 6-hourly times
    if day == n_days+1:
        endpoint=True
    else:
        endpoint=False
        
    times = make_8hourly_times(year, month, day, endpoint=False)
    times_py = pd.to_datetime(times).to_pydatetime()
    time_nums = cftime.date2num(times_py, units=time_var.units, calendar=time_var.calendar)
    

    # Create a dummy DataArray with the time dimension you want
    dummy_time = xr.DataArray(
        np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),  # shape: ("time", "f", "nj", "ni")
        coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
        dims=("time", "f", "nj", "ni"),
    )
    
    # Broadcast efreq along the time dimension
    da_broadcast = xr.broadcast(dummy_time, da.drop_vars(['TLON', 'TLAT']))[1]  # second element is original data broadcasted
    da_broadcast['TLON'] = da['TLON']
    da_broadcast['TLAT'] = da['TLAT']
    da_broadcast
    
    dummy_txy = xr.DataArray(
        np.empty((len(times), len(da.nj), len(da.ni))),
        coords={"time": times, "nj": da.nj, "ni": da.ni},
        dims=("time", "nj", "ni"),
    )
    
    vars_no_f = ["wave_sig_ht", "peak_period"]
    
    da_tf = xr.broadcast(
        dummy_txy,
        da[vars_no_f]
    )[1]
    
    dummy_tfxy = xr.DataArray(
        np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),
        coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
        dims=("time", "f", "nj", "ni"),
    )
    
    efreq_bt = xr.broadcast(dummy_tfxy, da["efreq"])[1]
    
    
    da_broadcast = xr.Dataset(
        {
            **da_tf.data_vars,
            "efreq": efreq_bt,
            "TLON": da["TLON"],
            "TLAT": da["TLAT"],
        },
        coords={
            "time": times,
            "f": da.f,
            "nj": da.nj,
            "ni": da.ni,
        },
    )

    
    

    # # Write time into unlimited dimension
    # for i, tnum in enumerate(time_nums):
    #     time_var[time_index] = tnum
    #     time_index += 1

        
    time_var[time_index:time_index + len(times)] = times
    # ------------------------------------------------------------------
    # wave_sig_ht(time, nj, ni)
    # ------------------------------------------------------------------
    npoints = len(times)
    wave_sig_ht_ext = xr.concat([da['wave_sig_ht']] * npoints, dim='time')
    
    wave_sig_ht_vals = wave_sig_ht_ext.transpose("time","nj","ni").values.astype("float32")
    
    # replace NaNs/Infs/extreme fill values
    # wave_sig_ht_vals[~np.isfinite(wave_sig_ht_vals)] = 0.0
    # wave_sig_ht_vals[wave_sig_ht_vals < 0] = 0.0  # optional, since spectra are non-negative
    
    wave_sig_ht_var[time_index:time_index + len(times), :, :] = wave_sig_ht_vals
    wave_sig_ht_var.setncatts(da_broadcast["wave_sig_ht"].attrs)
    
    # Peak period
    peak_period_ext = xr.concat([da['peak_period']] * npoints, dim='time')
    
    peak_period_vals = peak_period_ext.transpose("time","nj","ni").values.astype("float32")
    
    # replace NaNs/Infs/extreme fill values
    # peak_period_vals[~np.isfinite(peak_period_vals)] = 0.0
    # peak_period_vals[peak_period_vals < 0] = 0.0  # optional, since spectra are non-negative
    
    peak_period_var[time_index:time_index + len(times), :, :] = peak_period_vals
    peak_period_var.setncatts(da_broadcast["peak_period"].attrs)
    
    # ------------------------------------------------------------------
    # efreq(time, f, nj, ni)
    # ------------------------------------------------------------------
    efreq_ext = xr.concat([da['efreq']] * npoints, dim='time')
    
    efreq_vals = efreq_ext.transpose("time","f","nj","ni").values.astype("float32")
    
    # replace NaNs/Infs/extreme fill values
    # efreq_vals[~np.isfinite(efreq_vals)] = 0.0
    # efreq_vals[efreq_vals < 0] = 0.0  # optional, since spectra are non-negative
    
    efreq_var[time_index:time_index + len(times), :, :, :] = efreq_vals
    efreq_var.setncatts(da_broadcast["efreq"].attrs)

    ds_om3.close()
    time_index = time_index + len(times)

ds_example.close()
ds_new.close()
print(f"NetCDF file for month {year}-{month:02d} created successfully!")

In [ ]:
ds_tmp = xr.open_dataset(new_file, decode_times=False)
ds_tmp['wave_sig_ht'].isel(time=3).plot()

In [ ]:
ds_tmp['wave_sig_ht'].isel(time=-1).plot()

In [ ]:
time_nums

In [ ]:
day = 1
file_date = date(year, month, day) + timedelta(days=1)

file_path_om3 = file_path_om3_template.format(
    year=file_date.year,
    month=file_date.month,
    day=file_date.day
)
ds_om3 = xr.open_dataset(file_path_om3)
ds_om3

In [ ]:
ds_om3['HS'].plot()

In [ ]:
ds_tmp = xr.open_dataset('/scratch/ps29/nd0349/CICE_RUNS/wave-propagation-1deg/history/iceh.2010-01-31.nc', decode_times=False)

In [ ]:
ds_tmp['time'].plot()

In [ ]:
wave_sig_ht_vals[-1,50:100,:]

# Create a whole year of forcing

In [ ]:
example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_example = xr.open_dataset(example_file)
ds_example

In [ ]:
import calendar
import pandas as pd
import numpy as np
import xarray as xr
import cftime
from netCDF4 import Dataset
from datetime import date, timedelta

def year_month_to_output(year, month, base_year=2010):
    """
    Map (year, month) -> output index.
    
    month: 1–12
    """
    if not 1 <= month <= 12:
        raise ValueError("month must be in 1..12")

    return 12 * (year - base_year) + (month - 1)

def make_8hourly_times(year, month, day, endpoint=False):
    base = np.datetime64(f"{year}-{month:02d}-{day:02d}T00:00:00", "ns")
    # offsets = np.arange(0, 24, 6).astype("timedelta64[h]")
    if endpoint:
        offsets = np.linspace(0, 24, 5, endpoint=endpoint).astype("timedelta64[h]")
    else:
        offsets = np.linspace(0, 24, 4, endpoint=endpoint).astype("timedelta64[h]")
    times_ns = base + offsets
    return times_ns


### ------------------------------ YEAR TO PROCESS ------------------------------
year = 2023
month = 1
### -----------------------------------------------------------------------------
# Number of days in the month
# n_days = calendar.monthrange(year, month)[1]

# Open your daily file template
file_path_om3_template = (
    "/scratch/ps29/nd0349/access-om3/archive/"
    "IC4M8-MCW-100km_jra_iaf_2010/{output_dir}/"
    "access-om3.ww3.hi.{year:04d}-{month:02d}-{day:02d}-00000.nc"
)

# Prepare new NetCDF output
# new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}.nc"
new_file = (
    f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/"
    f"cice6-wim/cice6-wim_bretschneider_spec_{year}.nc"
)
ds_new = Dataset(new_file, "w", format="NETCDF4")

ds_example_roach = Dataset(example_file, "r")
ds_new.setncatts({attr: ds_example_roach.getncattr(attr) for attr in ds_example_roach.ncattrs()})

# Create dimensions (time unlimited)
ds_new.createDimension("time", None)

# We'll use first day to get static dimensions
ds_example = Dataset('/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/template.nc', "r")
ds_new.createDimension("f", ds_example.dimensions["f"].size)
ds_new.createDimension("nj", ds_example.dimensions["nj"].size)
ds_new.createDimension("ni", ds_example.dimensions["ni"].size)

# Create variables (simplified; you'll fill attributes as before)
time_var = ds_new.createVariable("time", "f8", ("time",), fill_value=False)
time_var.units = "hours since 1900-01-01 00:00:00"
time_var.calendar = "gregorian"

# Fill static variables from first day
# ------------------------------------------------------------------
# f (frequency axis)
# ------------------------------------------------------------------
# Create variable once
f_var = ds_new.createVariable("f", "f4", ("f",))
freqs, periods = get_ww3_freqs(nk=25)
f_var[:] = freqs.astype("float32")
#da_broadcast["f"].values.astype("float32")  # assign values

# Set attributes
f_var.long_name = "wave_frequency"
f_var.units = "s-1"
f_var.axis = "Hz"
f_var.standard_name = "wave_frequency"

# ------------------------------------------------------------------
# Grid (2-D only — collapse any broadcast dims)
# ------------------------------------------------------------------
TLON = ds_new.createVariable("TLON", "f4", ("nj", "ni"), fill_value=False)
TLON.standard_name = "longitude"
TLON.long_name = "longitude"
TLON.units = "degrees_east"
TLON._CoordinateAxisType = "Lon"
TLON[:] = ds_example.variables["TLON"][:]
# da_broadcast["TLON"].isel(
#     {d: 0 for d in da_broadcast["TLON"].dims if d not in ("nj", "ni")}
# ).values.astype("float32")

TLAT = ds_new.createVariable("TLAT", "f4", ("nj", "ni"), fill_value=False)
TLAT.standard_name = "latitude"
TLAT.long_name = "latitude"
TLAT.units = "degrees_north"
TLAT._CoordinateAxisType = "Lat"
TLAT[:] = ds_example.variables["TLAT"][:]
# da_broadcast["TLAT"].isel(
    # {d: 0 for d in da_broadcast["TLAT"].dims if d not in ("nj", "ni")}
# ).values.astype("float32")


wave_sig_ht_var = ds_new.createVariable(
    "wave_sig_ht",
    "f4",
    ("time", "nj", "ni"),
    chunksizes=(1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)
peak_period_var = ds_new.createVariable(
    "peak_period",
    "f4",
    ("time", "nj", "ni"),
    chunksizes=(1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)
efreq_var = ds_new.createVariable(
    "efreq",
    "f4",
    ("time", "f", "nj", "ni"),
    chunksizes=(1, 1, ds_new.dimensions["nj"].size, ds_new.dimensions["ni"].size),
    fill_value=False,
)



# Prepare time accumulator
all_times = []

time_index = 0   # 🔑 ONE counter for the whole year

for month in tqdm(range(1, 13)):
    output_dir = f"output{year_month_to_output(year, month, base_year=2010):03d}"
    
    n_days = calendar.monthrange(year, month)[1]

    for day in range(1, n_days + 1):

        # ---- file is offset by +1 day ----
        file_date = date(year, month, day) + timedelta(days=1)

        file_path_om3 = file_path_om3_template.format(
            output_dir=output_dir,
            year=file_date.year,
            month=file_date.month,
            day=file_date.day
        )

        ds_om3 = xr.open_dataset(file_path_om3)

        # ---- rename & subset ----
        ds_om3 = ds_om3.rename({
            "ny": "nj", "nx": "ni", "freq": "f",
            "lon": "TLON", "lat": "TLAT",
            "EF": "efreq", "HS": "wave_sig_ht", "FP0": "peak_period"
        })[["TLON", "TLAT", "wave_sig_ht", "peak_period", "efreq", "f"]]

        endpoint = (month == 12) and (day == 31)
        
        times = make_8hourly_times(year, month, day, endpoint=endpoint)
        times_py = pd.to_datetime(times).to_pydatetime()
        time_nums = cftime.date2num(
            times_py, units=time_var.units, calendar=time_var.calendar
        )
        if (month == 12) and (day == 31):
            print(endpoint)
            print(times)

        nt = len(times)

        # ---- write time ----
        time_var[time_index:time_index + nt] = times

        # ---- broadcast and write data ----
        da = ds_om3

        # Create a dummy DataArray with the time dimension you want
        dummy_time = xr.DataArray(
            np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),  # shape: ("time", "f", "nj", "ni")
            coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
            dims=("time", "f", "nj", "ni"),
        )
        
        # Broadcast efreq along the time dimension
        da_broadcast = xr.broadcast(dummy_time, da.drop_vars(['TLON', 'TLAT']))[1]  # second element is original data broadcasted
        da_broadcast['TLON'] = da['TLON']
        da_broadcast['TLAT'] = da['TLAT']

        
        dummy_txy = xr.DataArray(
            np.empty((len(times), len(da.nj), len(da.ni))),
            coords={"time": times, "nj": da.nj, "ni": da.ni},
            dims=("time", "nj", "ni"),
        )
        
        vars_no_f = ["wave_sig_ht", "peak_period"]
        
        da_tf = xr.broadcast(
            dummy_txy,
            da[vars_no_f]
        )[1]
        
        dummy_tfxy = xr.DataArray(
            np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),
            coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
            dims=("time", "f", "nj", "ni"),
        )
        
        efreq_bt = xr.broadcast(dummy_tfxy, da["efreq"])[1]
        
        
        da_broadcast = xr.Dataset(
            {
                **da_tf.data_vars,
                "efreq": efreq_bt,
                "TLON": da["TLON"],
                "TLAT": da["TLAT"],
            },
            coords={
                "time": times,
                "f": da.f,
                "nj": da.nj,
                "ni": da.ni,
            },
        )
        
        wave_sig_ht_ext = xr.concat([da['wave_sig_ht']] * nt, dim='time')
        wave_sig_ht_vals = wave_sig_ht_ext.transpose("time","nj","ni").values.astype("float32")

        wave_sig_ht_vals[~np.isfinite(wave_sig_ht_vals)] = 0.0
        wave_sig_ht_vals[wave_sig_ht_vals < 0] = 0.0

        wave_sig_ht_var[time_index:time_index + nt, :, :] = wave_sig_ht_vals

        peak_period_ext = xr.concat([da['peak_period']] * nt, dim='time')
        peak_period_vals = peak_period_ext.transpose("time","nj","ni").values.astype("float32")

        peak_period_vals[~np.isfinite(peak_period_vals)] = 0.0
        peak_period_vals[peak_period_vals < 0] = 0.0

        peak_period_var[time_index:time_index + nt, :, :] = peak_period_vals

        efreq_ext = xr.concat([da['efreq']] * nt, dim='time')
        efreq_vals = efreq_ext.transpose("time","f","nj","ni").values.astype("float32")

        efreq_vals[~np.isfinite(efreq_vals)] = 0.0
        efreq_vals[efreq_vals < 0] = 0.0

        efreq_var[time_index:time_index + nt, :, :, :] = efreq_vals

        # ---- advance global time index ----
        time_index += nt

        ds_om3.close()

ds_example.close()
ds_new.close()
print(f"NetCDF file for month {year}-{month:02d} created successfully!")

### Plot the forcing

In [ ]:
ds_forcing = xr.open_dataset(f'/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}.nc',
                            decode_times=False,
                            )
ds_forcing

In [ ]:
ds_forcing['wave_sig_ht'].isel(time=0).plot()

In [ ]:
ds_forcing['efreq'].isel(time=-1, f=12).plot()

In [ ]:
f_var.long_name = da_broadcast.coords["f"].attrs.get("long_name", "wave_frequency")

In [ ]:
from netCDF4 import Dataset

# Open the example file in read mode
example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_example = Dataset(example_file, "r")

# Create a new NetCDF file
new_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_2010-01-01_test.nc"
ds_new = Dataset(new_file, "w", format="NETCDF4")

# Copy global attributes
ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})

# Define new dimensions based on S_f_combined_ds
for dim_name, size in S_f_combined_ds.sizes.items():
    ds_new.createDimension(dim_name, size)

# Copy variables and assign data from S_f_combined_ds
for var_name, var in ds_example.variables.items():
    if var_name in S_f_combined_ds:
        # Create a new variable with the same datatype and dimensions
        new_var = ds_new.createVariable(var_name, var.datatype, var.dimensions)

        # Copy variable attributes
        new_var.setncatts({attr: var.getncattr(attr) for attr in var.ncattrs()})

        # Assign data from S_f_combined_ds
        new_var[:] = S_f_combined_ds.transpose("time", "f", "nj", "ni")[var_name].values

# Close the datasets
ds_example.close()
ds_new.close()

print(f"New NetCDF file '{new_file}' created successfully with data from S_f_combined_ds!")

In [ ]:
from glob import glob
from tqdm.notebook import tqdm


# files = sorted(glob('/scratch/tm70/ek4684/access-om3/archive/MCW_100km_jra_iaf_IC4_KPP/output000/access-om3.ww3.hi.*.nc'))
files = sorted(glob('/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_iaf_2010/output000/access-om3.ww3.hi.*.nc'))

for file in tqdm(files):
    # Open MCW output
    ds_om3 = xr.open_dataset(file)
    ds_om3 = ds_om3.rename({"ny": "nj", 
                            "nx": "ni", 
                            "freq": "f",
                            "lon": "TLON",
                            "lat": "TLAT",
                            "EF": "efreq",
                            "HS": "wave_sig_ht",
                            "FP0": "peak_period"
                           })
    freqs, periods = get_ww3_freqs(nk=25)
    xr_freqs = xr.DataArray(
        freqs,
        dims=("f",),
        # coords={"f": ds_om3.f},
        name='f',
        attrs={
            'units': 's-1',
            'description': 'Wave frequency',
            'long_name': 'wave_frequency',
        }
    )
    
    keep_vars = ["TLON", "TLAT", "wave_sig_ht", "peak_period", "efreq", "f"]
    
    ds_om3 = ds_om3.drop_vars(
        [v for v in ds_om3.data_vars if v not in keep_vars],
        errors="ignore",
    )
    ds_om3["f"] = xr_freqs
    ds_om3 = ds_om3.assign_coords(f=ds_om3.f, time=ds_om3.time)

    
    time = ds_om3.time - np.timedelta64(1, "D")
    year = time.dt.year.item()
    month = time.dt.month.item()
    day = time.dt.day.item()
    times = make_6hourly_times(year, month, day)

    da = ds_om3.copy()
    # Create a dummy DataArray with the time dimension you want
    dummy_time = xr.DataArray(
        np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),  # shape: ("time", "f", "nj", "ni")
        coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
        dims=("time", "f", "nj", "ni"),
    )
    
    # Broadcast efreq along the time dimension
    da_broadcast = xr.broadcast(dummy_time, da)[1]  # second element is original data broadcasted
    da_broadcast

    # Save NetCDF
    ds_example = Dataset(example_file, "r")
    
    # Create a new NetCDF file
    new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}-{day:02d}.nc"
    ds_new = Dataset(new_file, "w", format="NETCDF4")
    
    # Copy global attributes
    ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})
    
    # Define new dimensions based on S_f_combined_ds
    for dim_name, size in da_broadcast.sizes.items():
        ds_new.createDimension(dim_name, size)
        # if dim_name not in da_broadcast.data_vars:
            # var = ds_new.createVariable(dim_name, "f4" if dim_name=="f" else "i4", (dim_name,))
            # var[:] = da_broadcast.coords[dim_name].values
    
    
    for var_name, da_var in da_broadcast.data_vars.items():  # iterate over DataArray vars
        dims = da_var.dims  # these are correct: ('time','f','nj','ni')
        new_var = ds_new.createVariable(var_name, da_var.dtype, dims)
        # copy attributes if you have them
        for attr in da_var.attrs:
            new_var.setncattr(attr, da_var.attrs[attr])
        # assign values
        new_var[:] = da_var.values

    # After creating dimensions
    f_var = ds_new.createVariable("f", "f4", ("f",))
    f_var[:] = da_broadcast.coords["f"].values
    f_var.units = da_broadcast.coords["f"].attrs.get("units", "s-1")
    f_var.long_name = da_broadcast.coords["f"].attrs.get("long_name", "wave_frequency")
    
    # Close the datasets
    ds_example.close()
    ds_new.close()
    
    print(f"New NetCDF file '{new_file}' created successfully with data from da_broadcast!")

In [ ]:
import xarray as xr
import numpy as np
from glob import glob
from tqdm.notebook import tqdm
from netCDF4 import Dataset

# Collect all daily files for a given month
files = sorted(glob('/scratch/ps29/nd0349/access-om3/archive/IC4M8-MCW-100km_jra_iaf_2010/output000/access-om3.ww3.hi.*.nc'))

monthly_das = []  # List to store broadcasted daily DataArrays

for file in tqdm(files):
    ds_om3 = xr.open_dataset(file)
    ds_om3 = ds_om3.rename({
        "ny": "nj",
        "nx": "ni",
        "freq": "f",
        "lon": "TLON",
        "lat": "TLAT",
        "EF": "efreq",
        "HS": "wave_sig_ht",
        "FP0": "peak_period"
    })
    
    freqs, periods = get_ww3_freqs(nk=25)
    xr_freqs = xr.DataArray(freqs, dims=("f",), name="f",
                            attrs={"units": "s-1", "description": "Wave frequency", "long_name": "wave_frequency"})
    
    keep_vars = ["TLON", "TLAT", "wave_sig_ht", "peak_period", "efreq", "f"]
    ds_om3 = ds_om3.drop_vars([v for v in ds_om3.data_vars if v not in keep_vars], errors="ignore")
    ds_om3["f"] = xr_freqs
    ds_om3 = ds_om3.assign_coords(f=ds_om3.f, time=ds_om3.time)

    # Make 6-hourly times
    time = ds_om3.time - np.timedelta64(1, "D")
    year = time.dt.year.item()
    month = time.dt.month.item()
    day = time.dt.day.item()
    times = make_6hourly_times(year, month, day)
    
    # Broadcast data to 6-hourly times
    da = ds_om3.copy()
    dummy_time = xr.DataArray(
        np.empty((len(times), len(da.f), len(da.nj), len(da.ni))),
        coords={"time": times, "f": da.f, "nj": da.nj, "ni": da.ni},
        dims=("time", "f", "nj", "ni")
    )
    
    da_broadcast = xr.broadcast(dummy_time, da)[1]  # broadcast original data along new time
    monthly_das.append(da_broadcast)

# Concatenate all days along 'time'
monthly_ds = xr.concat(monthly_das, dim="time")

# Save to a single monthly NetCDF
monthly_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{year}-{month:02d}.nc"
monthly_ds.to_netcdf(monthly_file, format="NETCDF4")

print(f"Monthly NetCDF file '{monthly_file}' created successfully!")

In [ ]:
# cdo mergetime /g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_2010-01-*.nc /g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_2010-01.nc

### Convert from daily to monthly files

In [ ]:
# from netCDF4 import Dataset
# import numpy as np
# from glob import glob
# month = 1

# # Pattern to daily files
# daily_files = sorted(
#     glob(
#         f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/"
#         f"cice6-wim_bretschneider_spec_{year}-{month:02d}-*.nc"
#     )
# )

# # Open the first file to get dimensions and coords
# first_ds = Dataset(daily_files[0], "r")
# # f = first_ds.variables["f"][:]
# # nj = first_ds.variables["nj"][:]
# # ni = first_ds.variables["ni"][:]
# n_f = first_ds.dimensions['f'].size
# n_nj = first_ds.dimensions['nj'].size
# n_ni = first_ds.dimensions['ni'].size
# first_ds.close()

# # Count total time steps
# n_time_per_day = len(Dataset(daily_files[0]).variables["time"][:])
# n_days = len(daily_files)
# n_time_total = n_time_per_day * n_days

# # Create monthly NetCDF
# monthly_file = f"/g/data/ps29/.../cice6-wim_bretschneider_spec_{year}-{month:02d}.nc"
# ds_month = Dataset(monthly_file, "w", format="NETCDF4")

# # Create dimensions
# ds_month.createDimension("time", n_time_total)
# ds_month.createDimension("f", n_f)
# ds_month.createDimension("nj", n_nj)
# ds_month.createDimension("ni", n_ni)

# # Create variables
# time_var = ds_month.createVariable("time", "f8", ("time",))
# f_var = ds_month.createVariable("f", "f4", ("f",))
# nj_var = ds_month.createVariable("nj", "i4", ("nj",))
# ni_var = ds_month.createVariable("ni", "i4", ("ni",))

# wave_ht_var = ds_month.createVariable("wave_sig_ht", "f4", ("time", "f", "nj", "ni"))
# peak_period_var = ds_month.createVariable("peak_period", "f4", ("time", "f", "nj", "ni"))
# efreq_var = ds_month.createVariable("efreq", "f4", ("time", "f", "nj", "ni"))

# # Assign static coords
# f_var[:] = f
# nj_var[:] = nj
# ni_var[:] = ni

# # Fill time and variables
# t_index = 0
# for file in daily_files:
#     ds_day = Dataset(file, "r")
#     times = ds_day.variables["time"][:]
#     n_t = len(times)

#     wave_ht_var[t_index:t_index+n_t, :, :, :] = ds_day.variables["wave_sig_ht"][:]
#     peak_period_var[t_index:t_index+n_t, :, :, :] = ds_day.variables["peak_period"][:]
#     efreq_var[t_index:t_index+n_t, :, :, :] = ds_day.variables["efreq"][:]
    
#     time_var[t_index:t_index+n_t] = times  # copy time values
#     t_index += n_t
#     ds_day.close()

# ds_month.close()
# print(f"Monthly NetCDF created: {monthly_file}")

In [ ]:
# import subprocess

# daily_files = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_1958-01-*.nc"
# monthly_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_1958-01.nc"

# subprocess.run(["ncrcat", daily_files, monthly_file])

In [ ]:
cdo mergetime /g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_1958-01-*.nc               /g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_1958-01.nc

## Combining CICE6-WIM waves in ice with CAWCR in the open ocean

In [ ]:
file_path_cawcr = '/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/wave_output/ww3_om2_10deg_20190101.nc'

ds_cawcr = xr.open_dataset(file_path_cawcr)
ds_cawcr = ds_cawcr.isel(time=slice(-2,-1)) # select middle day (closest to average?)
ds_cawcr = ds_cawcr.rename({"dim_j": "nj", "dim_i": "ni"})

Hs_cawcr = ds_cawcr["hs"]  # Significant wave height (time, nj, ni)
fp_cawcr = ds_cawcr["fp"]  # Peak wave period (time, nj, ni)
fp_cawcr = np.clip(fp_cawcr, f.min(), None)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs_cawcr.time)
nj = len(Hs_cawcr.nj)
ni = len(Hs_cawcr.ni)

df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)
f_global = np.clip(f_global, f.min(), None)

# Compute the Bretschneider spectrum
S_f_cawcr = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f_cawcr[i_freq, :, :, :] = (5 / 16) * (Hs_cawcr ** 2) * fp_cawcr**(4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] / fp_cawcr) ** (-4))
    # print(np.nanmax(S_f_cawcr[i_freq, :, :, :]))

# Convert to xarray DataArray
S_f_cawcr_xr = xr.DataArray(
    S_f_cawcr,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="efreq"
)
S_f_cawcr_xr = S_f_cawcr_xr.assign_coords(time=S_f_cawcr_xr.time + pd.Timedelta(hours=2))

df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_cawcr
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_cawcr_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="Hs_test"
)


In [ ]:
# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_cawcr_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_cawcr.isel(nj=slice(0,100)).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_cawcr_xr - Hs_cawcr
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()
plt.show()


In [ ]:
S_f_cawcr_xr_copy = S_f_cawcr_xr.copy()
S_f_cawcr_xr_copy = xr.where(S_f_xr.notnull(), np.nan, S_f_cawcr_xr_copy)

S_f_combined_xr = xr.where(S_f_xr.isnull(), S_f_cawcr_xr_copy, S_f_xr)
S_f_combined_xr = xr.where(ds["tmask"] < 0.1, np.nan, S_f_combined_xr)
S_f_combined_xr = S_f_combined_xr.transpose("f","time", "nj", "ni")

S_f_combined_xr = xr.DataArray(
    np.tile(S_f_combined_xr.values, (1, 4, 1, 1)),  # Repeat across 4 time steps
    dims=["f", "time", "nj", "ni"],  # Define dimensions
    coords={"time": ds_ww3["time"], "f": S_f_combined_xr["f"], "nj": S_f_combined_xr["nj"], "ni": S_f_combined_xr["ni"]},
    name="efreq"
).astype(np.float32)

S_f_combined_xr = S_f_combined_xr.transpose("time", "nj", "ni","f")

S_f_combined_ds = xr.Dataset(
    {
        "efreq": (  # Define the variable efreq
            ["time","nj", "ni", "f"],
            S_f_combined_xr.values,  # The values with type cast
            {
                "standard_name": "power_spectral_density_of_surface_elevation",
                "long_name": "wave_elevation_spectrum",
                "units": "m² s",
                "coordinates": "f time nj ni",  # Reference to the coordinates
                "_FillValue": 9.96921e+36,  # Fill value
                "missing_value": 9.96921e+36,  # Missing value
                "globwave_name": "power_spectral_density_of_surface_elevation"  # Correct attribute name
            }
        ),
    },
    coords={
        "f": ("f", omega / (2 * np.pi), {"long_name": "frequency", "units": "Hz", "axis": "F", "standard_name": "sea_surface_wave_frequency"}),
    },

)


S_f_combined_ds = S_f_combined_ds.assign(TLON=ds['TLON'], TLAT=ds['TLAT'],time=ds_ww3["time"])
S_f_combined_ds


In [ ]:
S_f_combined_ds['efreq'].isel(f=6,time=1).plot()

In [ ]:
S_f_combined_ds

In [ ]:
# S_f_combined_xr.assign_coords(time=ds_ww3["time"]).astype(np.float32)

In [ ]:
# S_f_combined_xr = xr.DataArray(
#     np.tile(S_f_combined_xr.values, (1, 4, 1, 1)),  # Repeat across 4 time steps
#     dims=["f", "time", "nj", "ni"],  # Define dimensions
#     coords={"time": ds_ww3["time"], "f": S_f_combined_xr["f"], "nj": S_f_combined_xr["nj"], "ni": S_f_combined_xr["ni"]},
#     name="efreq"
# ).astype(np.float32)

In [ ]:
# S_f_combined_xr

In [ ]:
# df_global.shape

### Test plots of $H_s$ and $T_p$

In [ ]:
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
S_f_xr.isel(f=6).isel(nj=slice(0,100)).plot()

plt.title('CICE6-WIM')

# Plot Hs
plt.subplot(1, 3, 2)
S_f_cawcr_xr_copy.isel(f=6).isel(nj=slice(0,100)).plot()
plt.title('CAWCR')

# Plot the difference
plt.subplot(1, 3, 3)
S_f_combined_ds['efreq'].isel(f=6,time=0).isel(nj=slice(0,100)).plot()
plt.title('Combined')

plt.tight_layout()
plt.show()

In [ ]:
df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_combined_xr.transpose('f', 'time', 'nj', 'ni')
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_combined_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_ww3["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="Hs_test"
)
Hs_test_combined_xr = xr.where(ds["tmask"] < 0.1, np.nan, Hs_test_combined_xr)

In [ ]:
# Assuming Hs_test_xr, Hs_test_cawcr_xr, and Hs_test_combined_xr are xarray DataArrays
plt.figure(figsize=(10, 3))

# Determine the global colorbar limits (vmin and vmax)
vmin = min(Hs_test_xr.min(), Hs_test_cawcr_xr.min(), Hs_test_combined_xr.min())
vmax = max(Hs_test_xr.max(), Hs_test_cawcr_xr.max(), Hs_test_combined_xr.max())

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_xr.isel(nj=slice(0,100)).plot(vmin=vmin, vmax=vmax)
plt.title('CICE6-WIM')

# Plot Hs_test_cawcr_xr
plt.subplot(1, 3, 2)
Hs_test_cawcr_xr.isel(nj=slice(0,100)).plot(vmin=vmin, vmax=vmax)
plt.title('CAWCR')

# Plot the combined Hs_test_combined_xr
plt.subplot(1, 3, 3)
Hs_test_combined_xr.isel(nj=slice(0,100),time=0).plot(vmin=vmin, vmax=vmax)
plt.title('Combined')

# Add a colorbar for the entire figure
plt.tight_layout()
plt.show()

In [ ]:
# Assume df represents the frequency spectrum of the waves

# Reshaping and broadcasting the dataframe for frequency (df, which may represent energy density or similar)
df_reshaped_Tp = np.reshape(df, (nw, 1, 1, 1))
df_global_Tp = np.broadcast_to(df_reshaped_Tp, (nw, nt, nj, ni))

# Assuming S_f_combined_xr represents the wave energy density in the frequency domain
# Now calculate the energy for each frequency band (you might already have this, but here's the idea)
weighted_energy = df_global_Tp * S_f_combined_xr.transpose('f', 'time', 'nj', 'ni')


# Find the frequency index with the maximum energy (this is the peak frequency)
filled = weighted_energy.fillna(-np.inf)
peak_freq_index = filled.argmax(dim='f')
valid_mask = weighted_energy.isnull().all(dim='f')

# Calculate the peak period: peak period = 1 / peak frequency
# Assuming 'freqs' is an array of frequencies corresponding to the axis you're analyzing
freqs = np.broadcast_to(f.reshape(31, 1, 1, 1), (nw, nt, nj, ni))
peak_freqs = freqs[peak_freq_index]
Tp_test = 1 / peak_freqs

# Create an xarray DataArray for the peak period (Tp_test)
Tp_test_combined_xr = xr.DataArray(
    Tp_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="Tp_test"
)

# Apply tmask to exclude invalid regions (same as you did for Hs)
Tp_test_combined_xr = xr.where(ds["tmask"] < 0.1, np.nan, Tp_test_combined_xr)

# Plot the peak period
Tp_test_combined_xr.plot()

In [ ]:
# weighted_energy.dropna(dim=["f"], how="all")#.argmax()
# weighted_energy.argmax(dim='f')
# np.argmax(weighted_energy, axis=0)

# da = xr.DataArray(np.array([[0,2,3],[np.nan,np.nan,np.nan],[1,5,3]]))
# da.dropna(dim=["dim_0"], how="all").argmax()

In [ ]:
# freqs
# np.broadcast_to(freqs, (nw, nt, nj, ni))
# freqs_reshaped = freqs[:, None, None, None]  # shape becomes (31, 1, 1, 1)
# freqs_reshaped
# freqs_broadcasted = np.broadcast_to(f.reshape(31, 1, 1, 1), (nw, nt, nj, ni))

### Save to a NetCDF

In [ ]:
# # Save to a new NetCDF file
# save_filename = "/Users/noahday/GitHub/cice-dev/cice-dirs/input/CICE_data/forcing/access-om2_1deg/cice6-wim/cice6-wim_bretschneider_spec.nc"
# #f"/Users/noahday/GitHub/cice-dev/cice-dirs/input/CICE_data/forcing/access-om2_1deg/cice6-wim/cice6-wim_bretschneider_spec_{date_str}.nc"
# S_f_combined_ds.to_netcdf(save_filename, 
#                 mode='w',
#                 format = 'NETCDF4',
#                 unlimited_dims='time',
#                 )

# from netCDF4 import Dataset
# print("Bretschneider spectrum computed and saved.")

In [ ]:
from netCDF4 import Dataset

# Open the example file in read mode
example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_example = Dataset(example_file, "r")

# Create a new NetCDF file
new_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_2019-01-01.nc"
ds_new = Dataset(new_file, "w", format="NETCDF4")

# Copy global attributes
ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})

# Define new dimensions based on S_f_combined_ds
for dim_name, size in S_f_combined_ds.sizes.items():
    ds_new.createDimension(dim_name, size)

# Copy variables and assign data from S_f_combined_ds
for var_name, var in ds_example.variables.items():
    if var_name in S_f_combined_ds:
        # Create a new variable with the same datatype and dimensions
        new_var = ds_new.createVariable(var_name, var.datatype, var.dimensions)

        # Copy variable attributes
        new_var.setncatts({attr: var.getncattr(attr) for attr in var.ncattrs()})

        # Assign data from S_f_combined_ds
        new_var[:] = S_f_combined_ds.transpose("time", "f", "nj", "ni")[var_name].values

# Close the datasets
ds_example.close()
ds_new.close()

print(f"New NetCDF file '{new_file}' created successfully with data from S_f_combined_ds!")

In [ ]:
file_path = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{date_str}.nc"

ds_new = xr.open_dataset(file_path,decode_times=False)
ds_new['efreq']

ds_new['efreq'].isel(f=6,time=2).plot()

# Automate converting CICE6-WIM outputs to wave forcing files

In [ ]:
# Load the NetCDF file from CICE6-WIM
file_path = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/ice_output/iceh.2019-01-01.nc"

date_str = file_path.split("iceh.")[1].split(".nc")[0]
ds = xr.open_dataset(file_path)
ds['time'] = ds['time'] - pd.Timedelta(days=1)


# Extract variables
Hs = ds["wave_sig_ht"]  # Significant wave height (time, nj, ni)
Tp = ds["peak_period"]  # Peak wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs.time)
nj = len(Hs.nj)
ni = len(Hs.ni)


fmin = 0.042  # Minimum frequency (Hz)
fmax = 0.4            # Maximum frequency (Hz)

# Angular frequencies (rad/s)
om1 = 2 * np.pi * fmin
om2 = 2 * np.pi * fmax

# Frequency step size in rad/s
om_0 = (om2 - om1) / (nw - 1)

# Define frequency range (in Hz) based on omega values
omega = np.linspace(om1, om2, nw)  # Angular frequencies
f = np.array([[[omega / (2 * np.pi)]]])
df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
# Now we need to ensure that the `f` coordinate is broadcast to match the shape of the data
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)
    
def format_cice(ds):
    '''

    '''
    # df = f - np.concatenate(([0], f[0:-1]), axis=0)
    
    
    # Compute the Bretschneider spectrum
    S_f = np.empty([nw, nt, nj, ni])
    
    for i_freq, freq in enumerate(f[0][0][0]):
        S_f[i_freq, :, :, :] = (5 / 16) * (Hs ** 2) * Tp**(-4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] * Tp) ** (-4))
    
    # Convert to xarray DataArray
    S_f_xr = xr.DataArray(
        S_f,
        dims=["f", "time", "nj", "ni"],
        coords={"f": omega / (2 * np.pi), "time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
        name="efreq"
    )
    
    
    df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
    df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
    weighted_Sf = df_global*S_f
    # Sum over the first dimension (axis 0) to aggregate
    Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))
    
    Hs_test_xr = xr.DataArray(
        Hs_test,
        dims=["time", "nj", "ni"],
        coords={"time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
        name="Hs_test"
    )

    # Build dataset
    cice_xr = xr.Dataset({
        "efreq": S_f_xr,
        "Hs": Hs_test_xr
    })

    # Create new 6-hourly time coordinate
    base_date = pd.Timestamp(ds.time.values[0]).normalize()
    new_time = pd.date_range(start=base_date, periods=4, freq="6h")

    # Drop original single time dimension
    cice_xr = cice_xr.isel(time=0)  # remove the time dim (length=1)

    # Expand to new time dimension
    expanded = cice_xr.expand_dims(time=new_time)
    expanded = expanded.assign_coords(time=("time", new_time))

    # Add time attributes
    expanded["time"].attrs.update({
        "standard_name": "time",
        "long_name": "julian day (UT)",
        "axis": "T"
    })

    return expanded

cice_xr = format_cice(ds)
cice_xr


In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 3))

cice_xr['efreq'].isel(f=10, time=0).plot(ax=axs[0])
axs[0].set_title('efreq (f=10, time=0)')

cice_xr['Hs'].isel(time=0).plot(ax=axs[1])
axs[1].set_title('Hs (time=0)')

plt.tight_layout()
plt.show()

In [ ]:
# print(cice_xr.time)
# cice_xr['time']
# ds_ww3['time']

In [ ]:
# cawcr_xr['time']

In [ ]:
file_path_cawcr = '/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/wave_output/ww3_om2_10deg_20190101.nc'

# Load in CAWCR data and make timestamps match Lettie's example
# ds_ww3['time']
ds_cawcr = xr.open_dataset(file_path_cawcr)
ds_cawcr = ds_cawcr.sortby('time')
target_times = pd.date_range('2019-01-01', periods=4, freq='6h')  # e.g., 00, 06, 12, 18
ds_cawcr = ds_cawcr.sel(time=target_times, method='nearest')
ds_cawcr = ds_cawcr.rename({"dim_j": "nj", "dim_i": "ni"})


# ds_cawcr = xr.open_dataset(file_path_cawcr)
# ds_cawcr = ds_cawcr.isel(time=slice(-2,-1)) # select middle day (closest to average?)

# ds_cawcr['time'] = ds_cawcr['time'] - pd.Timedelta(hours=22) # Start at YYYY-MM-DDT00

def format_cawcr(ds_cawcr):
    '''

    '''
    Hs_cawcr = ds_cawcr["hs"]  # Significant wave height (time, nj, ni)
    fp_cawcr = ds_cawcr["fp"]  # Peak wave period (time, nj, ni)
    # fp_cawcr = np.clip(fp_cawcr, f.min(), None)
    fp_cawcr = xr.where(fp_cawcr < f.min(), f.min(), fp_cawcr)
    
    
    # Define parameters for frequency range
    nw = 31  # Number of frequency points
    nt = len(Hs_cawcr.time)
    nj = len(Hs_cawcr.nj)
    ni = len(Hs_cawcr.ni)
    
    df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
    f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
    f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)
    f_global = np.clip(f_global, f.min(), None)
    
    # Compute the Bretschneider spectrum
    S_f_cawcr = np.empty([nw, nt, nj, ni])
    
    for i_freq, freq in enumerate(f.squeeze()):
        S_f_cawcr[i_freq, :, :, :] = (5 / 16) * (Hs_cawcr ** 2) * fp_cawcr**(4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] / fp_cawcr) ** (-4))
        # print(np.nanmax(S_f_cawcr[i_freq, :, :, :]))
    
    # Convert to xarray DataArray
    S_f_cawcr_xr = xr.DataArray(
        S_f_cawcr,
        dims=["f", "time", "nj", "ni"],
        coords={"f": f.squeeze(), "time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
        name="efreq"
    )
    # S_f_cawcr_xr = S_f_cawcr_xr.assign_coords(time=S_f_cawcr_xr.time + pd.Timedelta(hours=4))
    # print(S_f_cawcr_xr.sum())
    
    df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
    df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
    weighted_Sf = df_global*S_f_cawcr
    Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))
    
    Hs_test_cawcr_xr = xr.DataArray(
        Hs_test,
        dims=["time", "nj", "ni"],
        coords={"time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
        name="Hs_test"
    )
    

     # Build dataset
    cawcr_xr = xr.Dataset({
        "efreq": S_f_cawcr_xr,
        "Hs": Hs_test_cawcr_xr
    })

    # Add time attributes
    cawcr_xr["time"].attrs.update({
        "standard_name": "time",
        "long_name": "julian day (UT)",
        "axis": "T"
    })
    
    return cawcr_xr

cawcr_xr = format_cawcr(ds_cawcr)
cawcr_xr


In [ ]:
# ds_cawcr['time']
# cawcr_xr['time']
# S_f_cawcr_xr
# ds_cawcr = xr.open_dataset(file_path_cawcr)
# ds_cawcr

In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 3))

cawcr_xr['efreq'].isel(time=-1, f=-1).plot(ax=axs[0])
cawcr_xr['Hs'].isel(time=0).plot(ax=axs[1])

plt.tight_layout()
plt.show()

# fp_cawcr.plot()

In [ ]:
cawcr_xr['Hs'].max()

In [ ]:
# xr.where(S_f_xr.notnull(), np.nan, S_f_cawcr_xr_copy)
cawcr_aligned = cawcr_xr['efreq'].reindex_like(cice_xr['efreq'], method='nearest')
cawcr_test = xr.where(cice_xr['efreq'].notnull(), cice_xr['efreq'], cawcr_aligned)

# xr.where(cice_xr['efreq'].notnull(), np.nan, cawcr_xr['efreq'])

In [ ]:
cawcr_test.isel(time=0, f=10).plot()

In [ ]:
# # Assuming Hs_test_xr and Hs are xarray DataArrays or similar
# plt.figure(figsize=(10, 3))

# # Plot Hs_test_xr
# plt.subplot(1, 3, 1)
# Hs_test_cawcr_xr.isel(nj=slice(0,100)).plot()
# plt.title('Hs_test_xr')

# # Plot Hs
# plt.subplot(1, 3, 2)
# Hs_cawcr.isel(nj=slice(0,100)).plot()
# plt.title('Hs')

# # Plot the difference
# plt.subplot(1, 3, 3)
# difference = Hs_test_cawcr_xr - Hs_cawcr
# difference.isel(nj=slice(0,100)).plot()
# plt.title('Difference (Hs_test_xr - Hs)')

# plt.tight_layout()
# plt.show()
# S_f_cawcr_xr

In [ ]:
def combine_cawcr_cice(S_f_cawcr_xr_copy, S_f_xr):
    '''
    Merge the domains of two wave spectra such that CAWCR is used in the open ocean and 
    CICE6-WIM is used in the ice-covered ocean.
       -  S_f_cawcr_xr_copy is a .copy() of a CACWCR xarray dataset
       - S_f_xr is a CICE6-WIM dataset
    Noah Day, 4 June 2025
    '''
    cawcr_aligned = S_f_cawcr_xr_copy.reindex_like(S_f_xr, method='nearest')
    S_f_combined_xr = xr.where(S_f_xr.notnull(), S_f_xr, cawcr_aligned)
    S_f_combined_xr = xr.where(ds["tmask"] < 0.1, np.nan, S_f_combined_xr)
    S_f_combined_xr = S_f_combined_xr.transpose("f","time", "nj", "ni")
    
    S_f_combined_xr = xr.DataArray(
        np.tile(S_f_combined_xr.values, (1, 1, 1, 1)),  # Repeat across 4 time steps
        dims=["f", "time", "nj", "ni"],  # Define dimensions
        coords={"time": S_f_combined_xr["time"], "f": S_f_combined_xr["f"], "nj": S_f_combined_xr["nj"], "ni": S_f_combined_xr["ni"]},
        name="efreq"
    ).astype(np.float32)
    
    
    # Transpose to (time, nj, ni, f) for final output
    S_f_combined_xr = S_f_combined_xr.transpose("time", "nj", "ni", "f")

    # Create final dataset
    S_f_combined_ds = xr.Dataset(
        {
            "efreq": (
                ["time", "nj", "ni", "f"],
                S_f_combined_xr.values,
                {
                    "standard_name": "power_spectral_density_of_surface_elevation",
                    "long_name": "wave_elevation_spectrum",
                    "units": "m² s",
                    "coordinates": "f time nj ni",
                    "_FillValue": 9.96921e+36,
                    "missing_value": 9.96921e+36,
                    "globwave_name": "power_spectral_density_of_surface_elevation"
                }
            )
        },
        coords={
            "f": (
                "f",
                omega / (2 * np.pi),
                {
                    "long_name": "frequency",
                    "units": "Hz",
                    "axis": "F",
                    "standard_name": "sea_surface_wave_frequency"
                }
            ),
            "time": S_f_combined_xr["time"],
            "nj": S_f_combined_xr["nj"],
            "ni": S_f_combined_xr["ni"]
        }
    )

    # Add spatial coordinates
    S_f_combined_ds = S_f_combined_ds.assign(TLON=ds["TLON"], TLAT=ds["TLAT"])

    # Recalculate Hs from the spectrum
    nt = len(S_f_combined_ds["time"])
    df_reshaped = np.reshape(df, (nw, 1, 1, 1))
    df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
    weighted_Sf = df_global * S_f_combined_ds["efreq"].transpose("f", "time", "nj", "ni")
    Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

    Hs_test_cawcr_xr = xr.DataArray(
        Hs_test,
        dims=["time", "nj", "ni"],
        coords={
            "time": S_f_combined_ds["time"],
            "nj": S_f_combined_ds["nj"],
            "ni": S_f_combined_ds["ni"]
        },
        name="Hs_test"
    )

    S_f_combined_ds["Hs"] = Hs_test_cawcr_xr


    return S_f_combined_ds


S_f_combined_ds = combine_cawcr_cice(cawcr_xr['efreq'], cice_xr['efreq'])
S_f_combined_ds



In [ ]:
fig, axs = plt.subplots(ncols=2, figsize=(8, 3))

S_f_combined_ds['efreq'].isel(f=10, time=0).plot(ax=axs[0])
axs[0].set_title('efreq (f=10, time=0)')

S_f_combined_ds['Hs'].isel(time=0).plot(ax=axs[1])
axs[1].set_title('Hs (time=0)')

plt.tight_layout()
plt.show()

In [ ]:
# date = S_f_combined_ds.time.dt.strftime('%Y-%m-%d').values[0]
# print(date)

### Save to NetCDF

In [ ]:
from netCDF4 import Dataset

# Open the example file in read mode
example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
ds_example = Dataset(example_file, "r")

date = cice_xr.time.dt.strftime('%Y-%m-%d').values[0]
# Create a new NetCDF file
new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{date}.nc"
ds_new = Dataset(new_file, "w", format="NETCDF4")

# Copy global attributes
ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})

# Define new dimensions based on S_f_combined_ds
for dim_name, size in S_f_combined_ds.sizes.items():
    ds_new.createDimension(dim_name, size)

# Copy variables and assign data from S_f_combined_ds
for var_name, var in ds_example.variables.items():
    if var_name in S_f_combined_ds:
        # Create a new variable with the same datatype and dimensions
        new_var = ds_new.createVariable(var_name, var.datatype, var.dimensions)

        # Copy variable attributes
        new_var.setncatts({attr: var.getncattr(attr) for attr in var.ncattrs()})

        # Assign data from S_f_combined_ds
        new_var[:] = S_f_combined_ds.transpose("time", "f", "nj", "ni")[var_name].values

# Close the datasets
ds_example.close()
ds_new.close()

print(f"New NetCDF file '{new_file}' created successfully with data from S_f_combined_ds!")

In [ ]:
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
freq_idx = 4
S_f_combined_ds['efreq'].isel(f=freq_idx,time=-1).isel(nj=slice(0,100)).plot()

plt.title(f'$S(f)$ at {np.round(1/f.reshape(-1)[freq_idx], 3)} s')

# Plot Hs
plt.subplot(1, 3, 2)
S_f_combined_ds['Hs'].isel(nj=slice(0,100),time=-1).plot()
plt.title('$H_s$')

# TODO: PLOT peak period!
# plt.subplot(1, 3, 3)
# S_f_combined_ds['efreq'].isel(f=8).isel(nj=slice(0,100)).plot()
# plt.title('Combined')

plt.tight_layout()
plt.show()

### Loop over files

In [ ]:
from netCDF4 import Dataset
YEAR = 2016
month_range = pd.date_range(f"{YEAR}-01-01", f"{YEAR}-12-31", freq="MS")

for month_start in month_range:
    month_end = (month_start + pd.offsets.MonthEnd(1)).date()
    date_list = pd.date_range(month_start, month_end)
    
    monthly_spectra = []
    
    for date in tqdm(date_list):
        # Load the NetCDF file from CICE6-WIM
        date_tmp = date.strftime('%Y-%m-%d') # e.g., 2019-01-01
        file_path = f"/g/data/ia40/cice-dirs/runs/waves-10/history/iceh.{date_tmp}.nc"
        
        date_str = file_path.split("iceh.")[1].split(".nc")[0]
        ds = xr.open_dataset(file_path)
        ds['time'] = ds['time'] - pd.Timedelta(days=1)
        
        
        # Extract variables
        Hs = ds["wave_sig_ht"]  # Significant wave height (time, nj, ni)
        Tp = ds["peak_period"]  # Peak wave period (time, nj, ni)
        
        # Define parameters for frequency range
        nw = 31  # Number of frequency points
        nt = len(Hs.time)
        nj = len(Hs.nj)
        ni = len(Hs.ni)
        
        fmin = 0.042  # Minimum frequency (Hz)
        fmax = 0.4    # Maximum frequency (Hz)
        
        # Angular frequencies (rad/s)
        om1 = 2 * np.pi * fmin
        om2 = 2 * np.pi * fmax
        
        # Frequency step size in rad/s
        om_0 = (om2 - om1) / (nw - 1)
        
        # Define frequency range (in Hz) based on omega values
        omega = np.linspace(om1, om2, nw)  # Angular frequencies
        f = np.array([[[omega / (2 * np.pi)]]])
        df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
        f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
        # Now we need to ensure that the `f` coordinate is broadcast to match the shape of the data
        f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)
    
        cice_xr = format_cice(ds)
    
        
        # 20190101
        date_tmp = date.strftime('%Y%m%d')
        file_path_cawcr = f"/g/data/ia40/cice-dirs/input/CICE_data/forcing/access-om2-10/CAWCR/{YEAR}/ww3_om2_10deg_{date_tmp}.nc"
    
        ds_cawcr = xr.open_dataset(file_path_cawcr)
        ds_cawcr = ds_cawcr.sortby('time')
        target_times = pd.date_range(str(ds_cawcr['time'][0].values)[:10], periods=4, freq='6h')  # e.g., 00, 06, 12, 18
        ds_cawcr = ds_cawcr.sel(time=target_times, method='nearest')
        ds_cawcr = ds_cawcr.rename({"dim_j": "nj", "dim_i": "ni"})
        
        cawcr_xr = format_cawcr(ds_cawcr)
    
        S_f_combined_ds = combine_cawcr_cice(cawcr_xr['efreq'], cice_xr['efreq'])
    
        for var in S_f_combined_ds.data_vars:
            data = S_f_combined_ds[var]
            if np.issubdtype(data.dtype, np.floating):
                # Fill NaNs
                data = data.fillna(0.0)
                # Replace Infs with 0 or another safe value
                data = data.where(np.isfinite(data), 0.0)
                S_f_combined_ds[var] = data
    
        # Open the example file in read mode
        example_file = "/g/data/ps29/nd0349/input/CICE_data/forcing/gx3/waves/WW3/ww3.20100101_efreq_remapgx3.nc"
        ds_example = Dataset(example_file, "r")
        
        date_tmp = cice_xr.time.dt.strftime('%Y-%m-%d').values[0]
        # Create a new NetCDF file
        new_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{date_tmp}.nc"
        # ds_new = Dataset(new_file, "w", format="NETCDF4")
        
        # Copy global attributes
        # ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})
        
        # Define new dimensions based on S_f_combined_ds
        # for dim_name, size in S_f_combined_ds.sizes.items():
        #     ds_new.createDimension(dim_name, size)
        
        # # Copy variables and assign data from S_f_combined_ds
        # for var_name, var in ds_example.variables.items():
        #     if var_name in S_f_combined_ds:
        #         # Create a new variable with the same datatype and dimensions
        #         new_var = ds_new.createVariable(var_name, var.datatype, var.dimensions)
        
        #         # Copy variable attributes
        #         new_var.setncatts({attr: var.getncattr(attr) for attr in var.ncattrs()})
        
        #         # Assign data from S_f_combined_ds
        #         new_var[:] = S_f_combined_ds.transpose("time", "f", "nj", "ni")[var_name].values
        
        monthly_spectra.append(S_f_combined_ds)
        # # Close the datasets
        # ds_example.close()
        # ds_new.close()
        
        # print(f"New NetCDF file '{new_file}' created successfully with data from S_f_combined_ds!")
    combined_month = xr.concat(monthly_spectra, dim="time")
    
    # Define output file path
    monthly_date_str = date_list[0].strftime("%Y%m")
    out_file = f"/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim/cice6-wim_bretschneider_spec_{monthly_date_str}.nc"
    
    with Dataset(out_file, "w", format="NETCDF4") as ds_new:
        # Copy global attributes
        ds_new.setncatts({attr: ds_example.getncattr(attr) for attr in ds_example.ncattrs()})
        
        # Define dimensions
        for dim_name, size in combined_month.sizes.items():
            ds_new.createDimension(dim_name, size)
        
        # Copy variables and assign data
        for var_name, var in ds_example.variables.items():
            if var_name in combined_month:
                new_var = ds_new.createVariable(var_name, var.datatype, var.dimensions)
                new_var.setncatts({attr: var.getncattr(attr) for attr in var.ncattrs()})
                new_var[:] = combined_month.transpose("time", "f", "nj", "ni")[var_name].values

ds_example.close()
print("✅ All monthly NetCDF files created successfully!")
        

In [ ]:
print(ds_yearly['time'])

### Combine monthly files to make a year

In [ ]:
YEAR = 2016
# Define the directory where monthly files are stored
input_dir = "/g/data/ps29/nd0349/input/CICE_data/forcing/access-om3-1deg/waves/cice6-wim"

# Create a file pattern to match all monthly files for the year
file_pattern = f"{input_dir}/cice6-wim_bretschneider_spec_{YEAR}??.nc"

# Get a sorted list of matching files
monthly_files = sorted(glob.glob(file_pattern))

# Load and concatenate them along the time dimension
ds_yearly = xr.open_mfdataset(monthly_files, 
                            #combine='by_coords',
                            decode_times=False, # 
                            decode_timedelta=True,
                            use_cftime=True, engine="netcdf4") 
time_raw = ds_yearly['time'].values
epoch = np.datetime64('1970-01-01T00:00:00')
time_fixed = epoch + time_raw.astype('timedelta64[ns]')

# Replace time variable with fixed datetime64 values
ds_yearly['time'] = ('time', time_fixed)

# Save the combined dataset to a new NetCDF file
output_file = f"{input_dir}/cice6-wim_bretschneider_spec_{YEAR}.nc"
ds_yearly.to_netcdf(output_file)

print(f"✅ Yearly NetCDF file created: {output_file}")

In [ ]:
print("NaNs in combined spectrum:", S_f_combined_ds.to_array().isnull().any().item())

In [ ]:
# Check for NaNs
if S_f_combined_ds.to_array().isnull().any():
    print(f"⚠️ NaNs detected in combined spectrum on {date}")
    # Optional: fill or raise error
    S_f_combined_ds = S_f_combined_ds.fillna(0)

print("NaNs in combined spectrum:", S_f_combined_ds.to_array().isnull().any().item())

In [ ]:
cawcr_xr['efreq']['time']

In [ ]:
cice_xr['efreq']['time']

In [ ]:
ds_cawcr['time']

In [ ]:
ds_cawcr = xr.open_dataset(file_path_cawcr)
ds_cawcr = ds_cawcr.sortby('time')
# target_times = pd.date_range('2019-01-01'
ds_cawcr['time']

# Extra Code

In [ ]:
# Load the NetCDF file from CICE6-WIM
file_path = "/Users/noahday/GitHub/cice-dev/cice-dirs/input/CICE_data/forcing/access-om2_1deg/cice6-wim/iceh.2019-01-01.nc"
date_str = file_path.split("iceh.")[1].split(".nc")[0]
ds = xr.open_dataset(file_path)

# Extract variables
Hs = ds["wave_sig_ht"]  # Significant wave height (time, nj, ni)
Tp = ds["peak_period"]  # Peak wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs.time)
nj = len(Hs.nj)
ni = len(Hs.ni)


fmin = 0.042  # Minimum frequency (Hz)
fmax = 0.4            # Maximum frequency (Hz)

# Angular frequencies (rad/s)
om1 = 2 * np.pi * fmin
om2 = 2 * np.pi * fmax

# Frequency step size in rad/s
om_0 = (om2 - om1) / (nw - 1)

# Define frequency range (in Hz) based on omega values
omega = np.linspace(om1, om2, nw)  # Angular frequencies
f = np.array([[[omega / (2 * np.pi)]]])

# df = f - np.concatenate(([0], f[0:-1]), axis=0)
df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)

f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)

# Now we need to ensure that the `f` coordinate is broadcast to match the shape of the data
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)

# Compute the Bretschneider spectrum
S_f = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f[i_freq, :, :, :] = (5 / 16) * (Hs ** 2) * Tp**(-4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] * Tp) ** (-4))

# Convert to xarray DataArray
S_f_xr = xr.DataArray(
    S_f,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="efreq"
)


df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f
# Sum over the first dimension (axis 0) to aggregate
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))
# 4*np.sqrt(np.dot(df_global, S_f[:,:,:,:]))

Hs_test_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds["time"], "nj": ds["nj"], "ni": ds["ni"]},
    name="Hs_test"
)


import matplotlib.pyplot as plt

# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs.where(ds['TLAT'] < -30, drop=True).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_xr - Hs
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()
plt.show()


## MOM6-CICE6-WW3 outputs

In [ ]:

file_path_om3 = '/Users/noahday/GitHub/cice-dev/cice-dirs/input/CICE_data/forcing/access-om2_1deg/cice6-wim/access-om3.ww3.hi.1958-01-02-00000.nc'

ds_om3 = xr.open_dataset(file_path_om3)
ds_om3 = ds_om3.isel(time=slice(-2,-1)) # select middle day (closest to average?)
ds_om3 = ds_om3.rename({"ny": "nj", "nx": "ni"})

Hs_om3 = ds_om3["HS"]  # Significant wave height (time, nj, ni)
Tp_om3 = ds_om3["T01"]  # Mean wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs_om3.time)
nj = len(Hs_om3.nj)
ni = len(Hs_om3.ni)

df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)

# Compute the Bretschneider spectrum
S_f_om3 = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f[i_freq, :, :, :] = (5 / 16) * (Hs_om3 ** 2) * Tp_om3**(-4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] * Tp_om3) ** (-4))
    print(np.nanmax(S_f_om3[i_freq, :, :, :]))

# Convert to xarray DataArray
S_f_cawcr_xr = xr.DataArray(
    S_f_om3,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds_om3["time"], "nj": ds_om3["nj"], "ni": ds_om3["ni"]},
    name="efreq"
)


df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_om3
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_om3_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_om3["time"], "nj": ds_om3["nj"], "ni": ds_om3["ni"]},
    name="Hs_test"
)



# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_om3_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_om3.isel(nj=slice(0,100)).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_om3_xr - Hs_om3
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()
plt.show()

S_f_om3_xr = S_f_om3_xr.assign_coords(time=S_f_om3_xr.time + pd.Timedelta(hours=2))

S_f_combined_xr = xr.where(S_f_xr == 0, S_f_om3_xr, S_f_xr)
S_f_combined_xr

In [ ]:
file_path_cawcr = '/Users/noahday/GitHub/cice-dev/cice-dirs/input/CICE_data/forcing/access-om2_1deg/cice6-wim/ww3_om2_10deg_20190101.nc'

ds_cawcr = xr.open_dataset(file_path_cawcr)
ds_cawcr = ds_cawcr.isel(time=slice(-2,-1)) # select middle day (closest to average?)
ds_cawcr = ds_cawcr.rename({"dim_j": "nj", "dim_i": "ni"})

Hs_cawcr = ds_cawcr["hs"]  # Significant wave height (time, nj, ni)
fp_cawcr = ds_cawcr["fp"]  # Peak wave period (time, nj, ni)

# Define parameters for frequency range
nw = 31  # Number of frequency points
nt = len(Hs_cawcr.time)
nj = len(Hs_cawcr.nj)
ni = len(Hs_cawcr.ni)

df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
f_reshaped = np.reshape(f, (nw, 1, 1, 1))  # Shape: (nw, 1, 1, 1)
f_global = np.broadcast_to(f_reshaped, (nw, nt, nj, ni))  # Shape: (nw, nt, nj, ni)

# Compute the Bretschneider spectrum
S_f_cawcr = np.empty([nw, nt, nj, ni])

for i_freq, freq in enumerate(f[0][0][0]):
    S_f_cawcr[i_freq, :, :, :] = (5 / 16) * (Hs_cawcr ** 2) * fp_cawcr**(4) * f_global[i_freq, :, :, :]**(-5) * np.exp( (-5/4) * (f_global[i_freq, :, :, :] / fp_cawcr) ** (-4))
    # print(np.nanmax(S_f_cawcr[i_freq, :, :, :]))

# Convert to xarray DataArray
S_f_cawcr_xr = xr.DataArray(
    S_f_cawcr,
    dims=["f", "time", "nj", "ni"],
    coords={"f": omega / (2 * np.pi), "time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="efreq"
)


df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_cawcr
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_cawcr_xr = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="Hs_test"
)



# Assuming Hs_test_xr and Hs are xarray DataArrays or similar
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_cawcr_xr.isel(nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_cawcr.isel(nj=slice(0,100)).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_cawcr_xr - Hs_cawcr
difference.isel(nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

plt.tight_layout()
plt.show()

S_f_cawcr_xr = S_f_cawcr_xr.assign_coords(time=S_f_cawcr_xr.time + pd.Timedelta(hours=2))

## Make regridder for CAWCR outputs onto the ACCESS grid

In [ ]:
import xesmf as xe
from netCDF4 import Dataset
import argparse
import os
import numpy             as np
import xarray            as xr
import pandas as pd
from datetime import datetime

######################################################
######################################################
def make_regridder(lon1, lat1, lon2, lat2, method, periodic, grdname,
                   lon1_b=None, lat1_b=None, lon2_b=None, lat2_b=None):
    '''
    make nearest neighbor xESMF regridder object.
    input:
    lon1: source longitudes (degrees)
    lat1: source latitudes  (degrees)
    lon2: target longitudes (degrees)
    lat2: target latitudes (degrees)
    method: regridding  method (bilinear, patch, conservative, nearest_s2d)
    periodic: True if periodic longitudes, false if not
    grdname: filename for regridder (Ugrid or Tgrid)
    '''
    if method != "conservative":
        # define grids for regridder
        grid1 = {'lon' : lon1, 'lat' : lat1}
        grid2 = {'lon' : lon2, 'lat' : lat2}


    else:
        LN,LT         = np.meshgrid(G_BRAN[lon_name].values,G_BRAN[lat_name].values)
        lontmp = np.tile(lon1,(length(lon1), length(lat1)))
        print(lontmp.shape)
        lattmp = np.tile(lat1,(length(lon1), length(lat1)))
        print(lattmp.shape)
        # conservative needs boundary lon/lat
        grid1 = {'lon'   : lon1,   'lat'   : lat1,
                 'lon_b' : lon1_b, 'lat_b' : lat1_b}

        grid2 = {'lon'   : lon2,   'lat'   : lat2,
                 'lon_b' : lon2_b, 'lat_b' : lat2_b}

    # make regridder
    # here specify reuse_weights=False to re-generate weight file.
    # if wanted to reuse file inteas of making int, 
    # check if file exists and change use_file_weights=True. 
    # see commented out example below 
    use_file_weights=True

    # check if regrid file exists.
    # If so, reuse file instead of regenerating.
    # if (os.path.isfile(blin_grid_name)):
    #     use_file_weights = True


    regridder = xe.Regridder(ds_in=grid1,ds_out=grid2,
                             method=method,
                             periodic=periodic,
                             filename=grdname,
                             reuse_weights=use_file_weights)



    return regridder


In [ ]:
import xarray as xr
import xesmf as xe

# Define regridder (assuming 'nj' and 'ni' are latitude and longitude)
regridder = xe.Regridder(S_f_cawcr_xr, S_f_xr, method="bilinear")  

# Apply regridding
S_f_cawcr_xr_regridded = regridder(S_f_cawcr_xr)

S_f_combined_xr = xr.where(S_f_xr == 0, S_f_cawcr_xr_regridded, S_f_xr)

In [ ]:
# pip install xesmf

# conda install -c conda-forge xesmf esmpy
# %conda update -c conda-forge xarray
%conda install -c conda-forge xarray=2023.6.0 cf_xarray xesmf

# import cf_xarray as cfxr

In [ ]:
df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_combined_xr
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_cawcr_xr_2 = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="Hs_test"
)
Hs_test_cawcr_xr_2.plot()

In [ ]:
S_f_xr < 0.00001

In [ ]:
# f_global.shape
# Hs_cawcr.plot()
S_f_cawcr_xr

In [ ]:
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_xr.isel(time=0,nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_cawcr.where(ds['TLAT'] < -30, drop=True).isel(time=0).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_xr - Hs
difference.isel(time=0,nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

In [ ]:
ds_cawcr

In [ ]:
df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
df

In [ ]:
# weighted_Sf = df[:, np.newaxis, np.newaxis, np.newaxis] * (S_f/S_f)


In [ ]:
f[0][0][0]

In [ ]:
np.nanmax(S_f)

In [ ]:
Hs.plot()

In [ ]:
S_f_xr.isel(f=0, time=0).plot()

In [ ]:
file_path = 
ds = xr.open_dataset(file_path)

## Make regridder for CAWCR outputs onto the ACCESS grid

In [ ]:
import xesmf as xe
from netCDF4 import Dataset
import argparse
import os
import numpy             as np
import xarray            as xr
import pandas as pd
from datetime import datetime

######################################################
######################################################
def make_regridder(lon1, lat1, lon2, lat2, method, periodic, grdname,
                   lon1_b=None, lat1_b=None, lon2_b=None, lat2_b=None):
    '''
    make nearest neighbor xESMF regridder object.
    input:
    lon1: source longitudes (degrees)
    lat1: source latitudes  (degrees)
    lon2: target longitudes (degrees)
    lat2: target latitudes (degrees)
    method: regridding  method (bilinear, patch, conservative, nearest_s2d)
    periodic: True if periodic longitudes, false if not
    grdname: filename for regridder (Ugrid or Tgrid)
    '''
    if method != "conservative":
        # define grids for regridder
        grid1 = {'lon' : lon1, 'lat' : lat1}
        grid2 = {'lon' : lon2, 'lat' : lat2}


    else:
        LN,LT         = np.meshgrid(G_BRAN[lon_name].values,G_BRAN[lat_name].values)
        lontmp = np.tile(lon1,(length(lon1), length(lat1)))
        print(lontmp.shape)
        lattmp = np.tile(lat1,(length(lon1), length(lat1)))
        print(lattmp.shape)
        # conservative needs boundary lon/lat
        grid1 = {'lon'   : lon1,   'lat'   : lat1,
                 'lon_b' : lon1_b, 'lat_b' : lat1_b}

        grid2 = {'lon'   : lon2,   'lat'   : lat2,
                 'lon_b' : lon2_b, 'lat_b' : lat2_b}

    # make regridder
    # here specify reuse_weights=False to re-generate weight file.
    # if wanted to reuse file inteas of making int, 
    # check if file exists and change use_file_weights=True. 
    # see commented out example below 
    use_file_weights=True

    # check if regrid file exists.
    # If so, reuse file instead of regenerating.
    # if (os.path.isfile(blin_grid_name)):
    #     use_file_weights = True


    regridder = xe.Regridder(ds_in=grid1,ds_out=grid2,
                             method=method,
                             periodic=periodic,
                             filename=grdname,
                             reuse_weights=use_file_weights)



    return regridder


In [ ]:
import xarray as xr
import xesmf as xe

# Define regridder (assuming 'nj' and 'ni' are latitude and longitude)
regridder = xe.Regridder(S_f_cawcr_xr, S_f_xr, method="bilinear")  

# Apply regridding
S_f_cawcr_xr_regridded = regridder(S_f_cawcr_xr)

S_f_combined_xr = xr.where(S_f_xr == 0, S_f_cawcr_xr_regridded, S_f_xr)

In [ ]:
# pip install xesmf

# conda install -c conda-forge xesmf esmpy
# %conda update -c conda-forge xarray
%conda install -c conda-forge xarray=2023.6.0 cf_xarray xesmf

# import cf_xarray as cfxr

In [ ]:
df_reshaped = np.reshape(df, (nw, 1, 1, 1)) 
df_global = np.broadcast_to(df_reshaped, (nw, nt, nj, ni))
weighted_Sf = df_global*S_f_combined_xr
Hs_test = 4 * np.sqrt(np.sum(weighted_Sf, axis=0))

Hs_test_cawcr_xr_2 = xr.DataArray(
    Hs_test,
    dims=["time", "nj", "ni"],
    coords={"time": ds_cawcr["time"], "nj": ds_cawcr["nj"], "ni": ds_cawcr["ni"]},
    name="Hs_test"
)
Hs_test_cawcr_xr_2.plot()

In [ ]:
S_f_xr < 0.00001

In [ ]:
# f_global.shape
# Hs_cawcr.plot()
S_f_cawcr_xr

In [ ]:
plt.figure(figsize=(10, 3))

# Plot Hs_test_xr
plt.subplot(1, 3, 1)
Hs_test_xr.isel(time=0,nj=slice(0,100)).plot()
plt.title('Hs_test_xr')

# Plot Hs
plt.subplot(1, 3, 2)
Hs_cawcr.where(ds['TLAT'] < -30, drop=True).isel(time=0).plot()
plt.title('Hs')

# Plot the difference
plt.subplot(1, 3, 3)
difference = Hs_test_xr - Hs
difference.isel(time=0,nj=slice(0,100)).plot()
plt.title('Difference (Hs_test_xr - Hs)')

In [ ]:
ds_cawcr

In [ ]:
df = f[0, 0, 0, :] - np.concatenate([np.array([0]), f[0, 0, 0, 0:-1]], axis=0)
df

In [ ]:
# weighted_Sf = df[:, np.newaxis, np.newaxis, np.newaxis] * (S_f/S_f)


In [ ]:
f[0][0][0]

In [ ]:
np.nanmax(S_f)

In [ ]:
Hs.plot()

In [ ]:
S_f_xr.isel(f=0, time=0).plot()